In [1]:
from data_handler import DataHandlerModule
from model_handler import ModelHandlerModule

config = {
    "seed":                        2,
    "data_name":                   "ieee_cis",
    "raw_dir":                     "./data",  # where CSVs live
    "sample":                      10000,     # small sample to verify
    "multi_relation":              True,
    "n_head":                      [2, 2],
    "n_head_agg":                  8,
    "feat_drop":                   0,
    "attn_drop":                   0,
    "train_ratio":                 0.1,
    "test_ratio":                  0.67,
    "emb_size":                    [64, 64],
    "lr":                          0.01,
    "weight_decay":                0.001,
    "epochs":                      50,        # small for quick check
    "valid_epochs":                10,
    "batch_size":                  1024,
    "patience":                    20,
    "cuda_id":                     0,
    "save_dir":                    "./results/ieee_cis",
    "apply_gan":                   False,
    "apply_graph_gan":             False,
    "apply_smote":                 False,    
    "apply_graph_smote":           False,
    "use_embedding_smote":         False, 
    "apply_contrastive_learning":  False,
    "apply_time_weight_decay":     False
}

In [2]:
import itertools
import copy

PRE_OPTIONS  = [None, "gan", "graph_gan", "smote", "graph_smote"]
POST_OPTIONS = [None, "contrastive_learning", "time_weight_decay"]

N_RUNS = 5

def build_config(base_config, pre, post, run_id):
  config = copy.deepcopy(base_config)
  
  config.update({
    "apply_gan": False,
    "apply_graph_gan": False,
    "apply_smote": False,
    "apply_graph_smote": False,
    "apply_contrastive_learning": False,
    "apply_time_weight_decay": False,
  })
  
  if pre == "gan":
    config["apply_gan"] = True
  elif pre == "graph_gan":
    config["apply_graph_gan"] = True
  elif pre == "smote":
    config["apply_smote"] = True
  
  if pre == "graph_smote":
    config["apply_graph_smote"] = True
    config["use_embedding_smote"] = True
  else:
    config["apply_graph_smote"] = False
    config["use_embedding_smote"] = False
    
  # Apply postprocessing
  if post == "contrastive_learning":
    config["apply_contrastive_learning"] = True
  elif post == "time_weight_decay":
    config["apply_time_weight_decay"] = True
  
  # Metadata
  config["run_id"] = run_id
  config["seed"] = run_id  # reproducibility
  config["experiment_name"] = f"pre_{pre or 'none'}__post_{post or 'none'}"

  return config    
    


In [ ]:
import os
import json
import pandas as pd
import itertools
import copy
from tqdm import tqdm
from utils import extract_embeddings

from post_training.contrastive_drag import run_contrastive_pipeline
from utils import test

RESULTS_DIR = "experiment_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

all_results = []

for pre, post in itertools.product(PRE_OPTIONS, POST_OPTIONS):

  exp_name = f"pre_{pre or 'none'}__post_{post or 'none'}"
  print(f"\n===== {exp_name} =====")

  for run in range(1, N_RUNS + 1):

    config = build_config(config, pre, post, run)

    # -------------------------
    # INIT
    # -------------------------
    data_handler = DataHandlerModule(config)
    model_handler = ModelHandlerModule(config, data_handler)
    
    # -------------------------
    #  EMBEDDING SMOTE PIPELINE
    # -------------------------
    if config["apply_graph_smote"]:

      print(">>> Running Embedding SMOTE Pipeline")

      # Pretrain encoder
      model_handler.pretrain_embeddings(epochs=5)

      # Extract embeddings
      embeddings = extract_embeddings(model_handler, data_handler)

      # Inject embeddings
      data_handler.embeddings = embeddings

      # Rebuild dataset WITH embeddings
      print("Rebuilding dataset with embedding GraphSMOTE...")
      data_handler = DataHandlerModule(config, embeddings=embeddings)

      # Rebuild model
      model_handler = ModelHandlerModule(config, data_handler)

    # -------------------------
    # TRAIN BASE MODEL
    # -------------------------
    auc, recall, f1 = model_handler.train()

    # Save baseline (optional but recommended)
    auc_base, recall_base, f1_base = auc, recall, f1
    
    

    # -------------------------
    #  POST: CONTRASTIVE LEARNING
    # -------------------------
    if config["apply_contrastive_learning"]:

      print(">>> Running Contrastive Post-Training")

      drag_model_original = model_handler.model
      graph = data_handler.dataset["graph"]

      contrastive_model = run_contrastive_pipeline(
        copy.deepcopy(drag_model_original),
        graph,
        config,
        data_handler=data_handler,
        run_3a=True,
        run_3b=False
      )

      contrastive_model.eval()

      auc, recall, f1, precision = test(
        contrastive_model,
        data_handler.dataset["test_loader"],
        model_handler.result,
        epoch=0,
        epoch_best=0,
        flag="test_contrastive"
      )

      print(f">>> Contrastive Results → AUC: {auc:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")

    # -------------------------
    # SAVE RESULT
    # -------------------------
    result = {
        "experiment": exp_name,
        "preprocessing": pre,
        "postprocessing": post,
        "run": run,
        "auc": float(auc),
        "recall": float(recall),
        "f1": float(f1),

        # Optional (very useful)
        "auc_base": float(auc_base),
        "recall_base": float(recall_base),
        "f1_base": float(f1_base)
    }

    filename = f"{exp_name}__run_{run}.json"
    filepath = os.path.join(RESULTS_DIR, filename)

    with open(filepath, "w") as f:
        json.dump(result, f, indent=4)

    all_results.append(result)

    print(f"Run {run}: AUC={auc:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

In [4]:
import os
import json
import pandas as pd

RESULTS_DIR = "experiment_results"

all_results = []

for filename in os.listdir(RESULTS_DIR):
    if filename.endswith(".json"):
        filepath = os.path.join(RESULTS_DIR, filename)

        with open(filepath, "r") as f:
            data = json.load(f)

            # 🔥 Ensure consistency (fix None issues if present)
            data["preprocessing"] = data.get("preprocessing") or "none"
            data["postprocessing"] = data.get("postprocessing") or "none"

            all_results.append(data)

df = pd.DataFrame(all_results)

print("Loaded runs:", len(df))
print("Unique experiments:", df["experiment"].nunique())

df.head()

Loaded runs: 75
Unique experiments: 15


,experiment,preprocessing,postprocessing,run,auc,recall,f1,auc_base,recall_base,f1_base
0,pre_gan__post_contrastive_learning,gan,contrastive_learning,1,0.754675,0.689500,0.499260,0.890321,0.704453,0.678494
1,pre_gan__post_contrastive_learning,gan,contrastive_learning,2,0.721689,0.491739,0.540183,0.896625,0.740155,0.669097
2,pre_gan__post_contrastive_learning,gan,contrastive_learning,3,0.816639,0.837067,0.418508,0.895058,0.746359,0.652081
3,pre_gan__post_contrastive_learning,gan,contrastive_learning,4,0.787020,0.771958,0.491154,0.893219,0.730621,0.681643
4,pre_gan__post_contrastive_learning,gan,contrastive_learning,5,0.817920,0.747290,0.526555,0.900485,0.727504,0.699080


In [5]:
df = pd.DataFrame(all_results)

summary = df.groupby("experiment").agg({
  "auc": ["min", "max", "mean", "std"],
  "recall": ["min", "max", "mean", "std"],
  "f1": ["min", "max", "mean", "std"]
})

summary.columns = [
  "auc_min", "auc_max", "auc_mean", "auc_std",
  "recall_min", "recall_max", "recall_mean", "recall_std",
  "f1_min", "f1_max", "f1_mean", "f1_std"
]

summary = summary.reset_index().sort_values(by="auc_mean", ascending=False)

summary.to_csv(f"{RESULTS_DIR}/summary.csv", index=False)

summary

,experiment,auc_min,auc_max,auc_mean,auc_std,recall_min,recall_max,recall_mean,recall_std,f1_min,f1_max,f1_mean,f1_std
2,pre_gan__post_time_weight_decay,0.893607,0.902876,0.899062,0.003529,0.718670,0.767452,0.738491,0.019851,0.632125,0.702336,0.679559,0.027319
1,pre_gan__post_none,0.890321,0.905277,0.896497,0.005885,0.704453,0.767452,0.734141,0.025709,0.632125,0.690711,0.673452,0.023567
13,pre_smote__post_none,0.860163,0.869829,0.864657,0.003505,0.682458,0.728611,0.706565,0.020548,0.600531,0.629503,0.614652,0.013362
14,pre_smote__post_time_weight_decay,0.860163,0.865410,0.863348,0.001982,0.682458,0.719416,0.703015,0.017017,0.600531,0.629503,0.615751,0.012912
7,pre_graph_smote__post_none,0.845700,0.856477,0.852000,0.004246,0.676303,0.722154,0.699623,0.018902,0.554212,0.592308,0.572675,0.014498
8,pre_graph_smote__post_time_weight_decay,0.845700,0.855143,0.851396,0.003618,0.654787,0.722154,0.695320,0.026472,0.554212,0.596154,0.573444,0.015840
5,pre_graph_gan__post_time_weight_decay,0.842271,0.854465,0.851322,0.005113,0.655924,0.719973,0.687547,0.025171,0.564759,0.596178,0.580530,0.011587
4,pre_graph_gan__post_none,0.842271,0.854465,0.851070,0.005046,0.618104,0.719973,0.676779,0.040415,0.564759,0.611673,0.585682,0.018339
10,pre_none__post_none,0.824897,0.853520,0.843922,0.011013,0.606256,0.730117,0.656570,0.050076,0.545636,0.621936,0.585384,0.027841
11,pre_none__post_time_weight_decay,0.824897,0.853131,0.843844,0.010929,0.614408,0.730117,0.660551,0.045674,0.545636,0.618114,0.584620,0.026612
